# Science LLM: Google Colab Master Pipeline (T4 GPU)

GitHub Repository: [greencube-dev-boop/Science-LLM](https://github.com/greencube-dev-boop/Science-LLM)

### Google Drive Organization:
- **PDFs and Text Files**: `drive/MyDrive/Science LLM/Dataset`
- **Checkpoints**: `drive/MyDrive/Science LLM/checkpoints`
- **Trained Model**: `drive/MyDrive/Science LLM/models/SCLLM-v<version>`
- **State & Config**: `drive/MyDrive/Science LLM/config/data.json` (auto-resume, versions, training runs, test logs)

In [ ]:
# 1. Verify GPU Allocation (Requires NVIDIA Tesla T4 16GB)
!nvidia-smi

In [ ]:
# 2. Clone the Science-LLM Repository into Colab
import os, sys

REPO_NAME = "Science-LLM"
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone https://github.com/greencube-dev-boop/Science-LLM.git /content/{REPO_NAME}

%cd /content/{REPO_NAME}
!git pull

if "/content/Science-LLM" not in sys.path:
    sys.path.append("/content/Science-LLM")

In [ ]:
# 3. Install Optimized Training Dependencies
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install -r requirements.txt

In [ ]:
# 4. Mount Google Drive & Initialize State Tracking via config_manager
from google.colab import drive
drive.mount('/content/drive')

from config_manager import ScienceLLMConfig

cfg = ScienceLLMConfig(base_dir="/content/drive/MyDrive/Science LLM", version="1.0.0")

print("Google Drive Structure Verified:")
print(f"- Dataset Directory:     {cfg.dataset_dir}")
print(f"- Checkpoints Directory: {cfg.checkpoints_dir}")
print(f"- Models Directory:      {cfg.models_dir}")
print(f"- State & Config File:   {cfg.data_json_path}")
print(f"- Active Target Model:   {cfg.state.get('model_name_tag')}")

## 5. Data Ingestion & Token Block Preparation
Place any science books, articles, or papers (`.pdf`, `.txt`) into `drive/MyDrive/Science LLM/Dataset`.
The pipeline parses them, strips reference boilerplate, preserves LaTeX math equations, packs them into 2048-token blocks, and updates `data.json`.

In [ ]:
from data_pipeline import DatasetBuilder

builder = DatasetBuilder(tokenizer_name="Qwen/Qwen2.5-3B", max_seq_length=2048, config=cfg)
records = builder.process_dataset_folder()

if records > 0:
    tokenized_dataset = builder.pack_sequences_for_dapt()
    print(f"Prepared {len(tokenized_dataset)} token blocks for Continued Pre-Training.")
else:
    print("No PDFs/text found in Dataset dir; will use curated SciQ STEM dataset as default for demonstration.")

## 6. Stage 1: Continued Pre-Training (DAPT) with Auto-Resume
Adapts the base model to raw scientific nomenclature and literature.
If Colab disconnects, re-running this cell automatically resumes training from the latest checkpoint stored in `data.json`.

In [ ]:
from train_continued_pretrain import run_training_unsloth, run_training_standard_peft, is_unsloth_available

print(f"Can resume from previous checkpoint: {cfg.state.get('checkpoints', {}).get('can_resume', False)}")
latest_ck = cfg.get_resume_checkpoint(stage="stage1_dapt")
if latest_ck:
    print(f"Resuming from: {latest_ck}")

if is_unsloth_available():
    run_training_unsloth(cfg, model_name="Qwen/Qwen2.5-3B", epochs=1, batch_size=2, grad_accum=8, lr=1e-4)
else:
    run_training_standard_peft(cfg, model_name="Qwen/Qwen2.5-3B", epochs=1, batch_size=1, grad_accum=16, lr=1e-4)

## 7. Stage 2: Supervised Fine-Tuning (SFT) & Scientific Reasoning
Loads the Stage 1 adapter and fine-tunes the model on step-by-step scientific problem solving with prompt completion masking.

In [ ]:
from train_sft import train_sft

train_sft(cfg, base_model_name="Qwen/Qwen2.5-3B", epochs=2, batch_size=2, grad_accum=8, lr=2e-4)

## 8. Export Standalone Merged Model: `SCLLM-v<version>`
Merges the trained LoRA adapter into 16-bit base weights and saves directly to:
`drive/MyDrive/Science LLM/models/SCLLM-v<version>`
Synchronizes model information into `data.json`.

In [ ]:
from export_and_evaluate import merge_and_export

merge_and_export(cfg, base_model_name="Qwen/Qwen2.5-3B", version="1.0.0")
print(f"Model export complete at: {cfg.get_model_save_dir('1.0.0')}")

## 9. Benchmark Evaluation & Interactive Science Testing
Tests the exported model on scientific domain prompts (CRISPR, quantum wave packets, organic synthesis) and logs output to `data.json`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from export_and_evaluate import evaluate_and_log

model_path = str(cfg.get_model_save_dir('1.0.0'))
print(f"Loading {model_path} for evaluation...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

evaluate_and_log(cfg, model, tokenizer, device="cuda")
print("Evaluation complete. Results logged to data.json.")